# 00 · Setup: architecture, tools and metric definitions

This notebook records the system as it starts: what each layer is, why each tool was chosen, and exactly how every metric in later notebooks is defined and measured. Later phases compare against the definitions here, so they don't change mid-project.

All numbers in these notebooks are read from saved runs in `results/`, never typed in by hand.

## Architecture

```
Client (CrewAI agent / Locust / smoke test)
   │
   ▼
API gateway      FastAPI, :8080   request ID, timing, metrics; harnesses in Phase 3
   │
   ▼
NGINX            :80 (internal)   load balancer; one upstream now, several in Phase 6
   │
   ▼
vLLM             :8000 (internal) model on the GPU: batching, KV cache, token generation
   │
   └── /metrics ──► Prometheus :9090 ──► Grafana :3000
gateway /metrics ─┘
```

Only the gateway, Prometheus and Grafana are published, and only on `127.0.0.1` of the GPU box. They are reached from a laptop through an SSH tunnel, so nothing is exposed to the internet.

## Tool choices

| Layer | Choice | Why |
|---|---|---|
| Application | CrewAI agent | Produces realistic multi-step LLM traffic, not just single requests |
| Gateway | Custom FastAPI | Every check, log line and timer is visible code, which makes harnesses and tracing easy to explain |
| Load balancer | NGINX | Standard, simple config; `upstream` block scales from 1 to N vLLM servers without changing anything else |
| Inference engine | vLLM | Paged attention + continuous batching, OpenAI-compatible API, rich Prometheus metrics |
| Model | Qwen2.5-7B-Instruct | ~8B class, no gated access, fits a 24–48 GB GPU with room for KV cache |
| Metrics | Prometheus + Grafana | vLLM and the gateway expose Prometheus metrics natively |
| Load testing | Locust + a traffic agent | Locust for controlled ramps, the agent for realistic workloads |
| KV cache | Mooncake | KV cache storage and transfer beyond a single GPU's memory |
| GPU sharing | HAMi | Slices GPUs between workloads on Kubernetes |
| GPU hosting | Vast.ai VM instances | Low prices, per-second billing, full VMs so Docker and Kubernetes work |

## Metric definitions

Every phase reports these, measured the same way.

| Metric | Definition | How it is measured |
|---|---|---|
| **TTFT** (time to first token) | Request sent → first generated token received | Client-side, streaming; also vLLM's `time_to_first_token_seconds` histogram |
| **Prefill latency** | Time the engine spends processing the prompt before decoding | vLLM's prefill-time histogram; approximately TTFT minus queueing and network |
| **ITL** (inter-token latency) | Average gap between consecutive output tokens | (end-to-end − TTFT) / (output tokens − 1) |
| **End-to-end latency** | Request sent → last token received | Client-side; also gateway `gateway_request_duration_seconds` |
| **P50 / P95 / P99** | The latency that 50 / 95 / 99 % of requests are at or below | Nearest-rank percentile over all requests in a run (`results_logger.summarize`) |
| **Throughput** | Output tokens per second, and requests per second, across all users | Total output tokens (or completed requests) / run duration |
| **KV cache usage** | Share of the KV cache blocks in use | vLLM's KV cache usage gauge |
| **Running / waiting** | Requests being generated vs. queued | vLLM's `num_requests_running` / `num_requests_waiting` |
| **Preemptions** | Running requests evicted because KV cache memory ran out | vLLM's preemption counter |
| **Error rate** | Failed requests / total requests | Non-2xx responses and timeouts at the client and gateway |
| **Cost per 1M output tokens** | GPU $/hour ÷ output tokens per hour × 1,000,000 | `GPU_HOURLY_PRICE_USD` from `.env` and measured throughput |

Exact vLLM metric names vary between versions; they are confirmed against the live `/metrics` endpoint in Phase 1.

## Experiment template

Every experiment in every notebook records:

1. **Hypothesis**: what should change, and why
2. **Setup**: hardware, model, the exact `vllm/configs/*.yaml` used
3. **Load**: users, spawn rate, prompt set, duration
4. **Results**: the metrics above, read from `results/`
5. **What happened**: Grafana panels and one representative trace
6. **Conclusion**: was the hypothesis right, and what is the next bottleneck

In [1]:
import json
import sys
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs

## The prompt set

24 synthetic prompts in `loadtest/prompts.json`, in four groups that stress different phases of inference:

- **short_qa**: tiny prompt, tiny answer. Mostly overhead and scheduling.
- **summarization**: long prompt, medium answer. Prefill-heavy.
- **multi_turn**: growing conversation history, as an agent sends. Exercises prefix reuse.
- **long_generation**: short prompt, long answer. Decode-heavy.

Token counts below are estimates (characters ÷ 4); real counts come from vLLM's `usage` field.

In [2]:
prompts = json.loads((ROOT / "loadtest" / "prompts.json").read_text())

groups = defaultdict(list)
for p in prompts:
    chars = sum(len(m["content"]) for m in p["messages"])
    groups[p["category"]].append((chars // 4, p["max_tokens"]))

print(f"{'category':<16}{'prompts':>8}{'≈ prompt tokens (min–max)':>28}{'max output tokens':>20}")
for category, rows in groups.items():
    lengths, outputs = [r[0] for r in rows], [r[1] for r in rows]
    output_range = f"{min(outputs)}–{max(outputs)}" if min(outputs) != max(outputs) else str(outputs[0])
    print(f"{category:<16}{len(rows):>8}{f'{min(lengths)}–{max(lengths)}':>28}{output_range:>20}")
print(f"{'total':<16}{len(prompts):>8}")

category         prompts   ≈ prompt tokens (min–max)   max output tokens
short_qa               8                       15–22                  64
summarization          6                    449–1097                 384
multi_turn             5                      71–147             192–384
long_generation        5                       26–36                 768
total                 24


## Baseline engine configuration

Experiments copy this file and change one value at a time.

In [3]:
print((ROOT / "vllm" / "configs" / "baseline.yaml").read_text())

# Baseline vLLM engine arguments. Keys are `vllm serve` flags without the "--".
# The model and served name are set in docker-compose.yml from .env.
#
# To run an experiment: copy this file (e.g. max_num_seqs_128.yaml), change ONE
# value, and start the stack with VLLM_CONFIG=max_num_seqs_128.
host: 0.0.0.0
port: 8000
gpu-memory-utilization: 0.90
max-model-len: 8192



## First run on the GPU

The stack was brought up on a rented **RTX 5090 (32 GB)** with `scripts/setup_gpu_box.sh`, which ended by sending one streaming request through gateway → NGINX → vLLM and saving it to `results/00_setup/`.

The image ran as `vllm/vllm-openai:latest`, which resolved to digest `sha256:8a69ffad015f…`, the same image as the `v0.30.0` tag. It is now pinned to `v0.30.0` so every later phase runs on exactly this version.

In [4]:
runs = load_runs("00_setup")
if not runs:
    print("No runs yet: run scripts/setup_gpu_box.sh on the GPU box, then copy results/ back.")
for run in runs:
    m, env = run["metrics"], run["environment"]
    gpu = env["gpus"][0] if env["gpus"] else {"name": "no GPU", "memory_mib": 0}
    print(f"run          {run['run_id']}")
    print(f"hardware     {gpu['name']}, {gpu['memory_mib'] / 1024:.0f} GB, driver {gpu.get('driver')}")
    print(f"model        {run['config'].get('model')}  (config: {run['config'].get('vllm_config')})")
    print(f"code         git {env['git_commit'][:7]}{' (dirty)' if env['git_dirty'] else ''}")
    print()
    print(f"TTFT         {m['ttft_s'] * 1000:.0f} ms")
    print(f"end-to-end   {m['e2e_s'] * 1000:.0f} ms")
    print(f"inter-token  {m['itl_s'] * 1000:.1f} ms  (≈ {1 / m['itl_s']:.0f} tokens/s while decoding)")
    print(f"tokens       {m['prompt_tokens']} in / {m['output_tokens']} out")

run          20261003T034035Z_smoke_test
hardware     NVIDIA GeForce RTX 5090, 32 GB, driver 580.95.05
model        Qwen/Qwen2.5-7B-Instruct  (config: baseline)
code         git acce6ba

TTFT         281 ms
end-to-end   878 ms
inter-token  9.6 ms  (≈ 104 tokens/s while decoding)
tokens       44 in / 63 out


### What vLLM reported at startup

These lines from the vLLM log (saved in `results/00_setup/vllm_startup.log`) set the limits the later phases will run into. The KV cache is what holds every active conversation, so its size caps how many requests can run at once.

In [5]:
import re

log = (ROOT / "results" / "00_setup" / "vllm_startup.log").read_text()

def grab(pattern):
    match = re.search(pattern, log)
    return match.group(1) if match else "?"

facts = {
    "weights on disk": grab(r"Checkpoint size: ([\d.]+ GiB)"),
    "weights download time": f'{float(grab(r"downloading weights for \S+: ([\d.]+) seconds")):.0f} s',
    "weights in GPU memory": grab(r"Model loading took ([\d.]+ GiB) memory"),
    "memory left for KV cache": grab(r"Available KV cache memory: ([\d.]+ GiB)"),
    "KV cache capacity": grab(r"GPU KV cache size: ([\d,]+) tokens") + " tokens",
    "max requests at 8,192 tokens each": grab(r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)x"),
}
for name, value in facts.items():
    print(f"{name:<36}{value}")

weights on disk                     14.19 GiB
weights download time               35 s
weights in GPU memory               14.29 GiB
memory left for KV cache            11.19 GiB
KV cache capacity                   209,472 tokens
max requests at 8,192 tokens each   25.57


Of the 32 GB on the card, about 14.3 GB holds the model weights and about 11.2 GB is left for the KV cache, enough for roughly 209k tokens in flight. If every request used the full 8,192-token context, only about 25 could run at once. Real requests are much shorter, so many more will fit; Phase 4 finds the actual breaking point.

## Phase 0 exit criteria

- [x] Repo layout, `docker-compose.yml`, gateway, NGINX, Prometheus and Grafana config
- [x] Gateway pass-through with request IDs and metrics, covered by tests
- [x] 24 synthetic prompts in `loadtest/prompts.json`
- [x] Results logger saving to `results/<phase>/<run_id>.json`
- [x] One-command setup script for the GPU box
- [x] `docker compose up` brings up the full stack on the GPU box (RTX 5090), all Prometheus targets are up, and the smoke test passes